In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
import wandb
import warnings
warnings.filterwarnings('ignore')

## Configuration & WandB Initialization

In [ ]:
import torch
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch W&B API key securely from Kaggle secrets
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WanDB_API_KEY")

# Log in to W&B using the secret key
wandb.login(key=wandb_api_key)

# W&B Init (APNA ROLL NO DAAL)
wandb.init(
    project="DL-23f1000967-notebook-t22026",
    name="EDA_tfidf_baseline",
    config={
        "method": "EDA + TF-IDF + Cosine Similarity",
        "description": "Baseline model using TF-IDF embeddings"
    }
)

# # Access config and device
# config = wandb.config
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Using device: {device}")


## Data Loading

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample_sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain columns: {train.columns.tolist()}")
print(f"\nSample train row:")
print(train.iloc[0])

## Exploratory Data Analysis

In [ ]:
# Answer distribution
answer_counts = train['answer'].value_counts().sort_index()
print("\nAnswer Distribution:")
print(answer_counts)

In [ ]:
# Plot answer distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [ ]:
# Bar plot
axes[0].bar(answer_counts.index, answer_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
axes[0].set_xlabel('Answer Label')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Answers in Training Data')
axes[0].set_xticks(['A', 'B', 'C', 'D', 'E'])

In [ ]:
# Pie chart
axes[1].pie(answer_counts.values, labels=answer_counts.index, autopct='%1.1f%%', 
           colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
axes[1].set_title('Answer Distribution (Percentage)')

plt.tight_layout()
plt.savefig('answer_distribution.png', dpi=150, bbox_inches='tight')
wandb.log({"answer_distribution": wandb.Image('answer_distribution.png')})
plt.show()

In [ ]:
# Text length analysis
train['prompt_length'] = train['prompt'].str.len()
train['prompt_words'] = train['prompt'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['prompt_length'], bins=50, color='#4ECDC4', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Prompt Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Prompt Length')
axes[0].axvline(train['prompt_length'].mean(), color='red', linestyle='--', label=f'Mean: {train["prompt_length"].mean():.0f}')
axes[0].legend()

axes[1].hist(train['prompt_words'], bins=50, color='#45B7D1', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Prompt Length (words)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Prompt Word Count')
axes[1].axvline(train['prompt_words'].mean(), color='red', linestyle='--', label=f'Mean: {train["prompt_words"].mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('prompt_length_analysis.png', dpi=150, bbox_inches='tight')
wandb.log({"prompt_length_analysis": wandb.Image('prompt_length_analysis.png')})
plt.show()

In [ ]:
# Option length analysis
for opt in ['A', 'B', 'C', 'D', 'E']:
    train[f'{opt}_length'] = train[opt].str.len()

option_lengths = [train[f'{opt}_length'].mean() for opt in ['A', 'B', 'C', 'D', 'E']]

plt.figure(figsize=(10, 5))
plt.bar(['A', 'B', 'C', 'D', 'E'], option_lengths, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'])
plt.xlabel('Option Label')
plt.ylabel('Average Length (characters)')
plt.title('Average Option Length by Label')
plt.savefig('option_length_analysis.png', dpi=150, bbox_inches='tight')
wandb.log({"option_length_analysis": wandb.Image('option_length_analysis.png')})
plt.show()

In [ ]:
# Log EDA stats to W&B
wandb.log({
    "train_size": len(train),
    "test_size": len(test),
    "avg_prompt_length": train['prompt_length'].mean(),
    "avg_prompt_words": train['prompt_words'].mean(),
    "min_prompt_length": train['prompt_length'].min(),
    "max_prompt_length": train['prompt_length'].max()
})

## Data Preprocessing



In [ ]:
import re

def clean_text(text):
    """Clean and preprocess text data"""
    if pd.isna(text):
        return ""
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove special characters but keep spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
# Apply cleaning
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train[f'{col}_clean'] = train[col].apply(clean_text)
    test[f'{col}_clean'] = test[col].apply(clean_text)

print("Text cleaning completed!")
print(f"\nSample cleaned prompt: {train['prompt_clean'].iloc[0][:200]}...")

## TF-IDF Vectorization

In [ ]:
# Combine prompt and each option for TF-IDF
def create_combined_text(row, option):
    """Combine prompt with option for TF-IDF"""
    return f"{row['prompt_clean']} {row[f'{option}_clean']}"

In [ ]:
# Create TF-IDF vectorizer
# from sklearn.feature_extraction.text import (
#     TfidfVectorizer,
#     ENGLISH_STOP_WORDS
# )
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english'
)

In [ ]:
# Fit on all text data
all_texts = []
for opt in ['A', 'B', 'C', 'D', 'E']:
    all_texts.extend(train[f'{opt}_clean'].tolist())
    all_texts.extend(test[f'{opt}_clean'].tolist())
all_texts.extend(train['prompt_clean'].tolist())
all_texts.extend(test['prompt_clean'].tolist())

tfidf.fit(all_texts)
print(f"TF-IDF vocabulary size: {len(tfidf.vocabulary_)}")

## Baseline Model

In [ ]:
def get_tfidf_predictions(df, tfidf, is_train=False):
    """
    Get predictions using TF-IDF cosine similarity between prompt and each option
    Returns top 3 predictions in ranked order
    """
    predictions = []
    
    for idx, row in df.iterrows():
        # Get prompt embedding
        prompt_vec = tfidf.transform([row['prompt_clean']])
        
        # Calculate similarity with each option
        similarities = {}
        for opt in ['A', 'B', 'C', 'D', 'E']:
            opt_vec = tfidf.transform([row[f'{opt}_clean']])
            sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
            similarities[opt] = sim
        
        # Sort by similarity and get top 3
        sorted_options = sorted(similarities.keys(), key=lambda x: similarities[x], reverse=True)
        top_3 = ' '.join(sorted_options[:3])
        predictions.append(top_3)
    
    return predictions

In [ ]:
# Get predictions for train (for validation)
train_predictions = get_tfidf_predictions(train, tfidf, is_train=True)

In [ ]:
# Get predictions for test
test_predictions = get_tfidf_predictions(test, tfidf, is_train=False)

print(f"\nSample predictions:")
for i in range(5):
    print(f"Q{i+1}: {test_predictions[i]}")

## MAP@3 Evaluation

In [ ]:
def map_at_3(actual, predicted):
    """
    Calculate Mean Average Precision at 3 (MAP@3)
    
    Args:
        actual: List of actual answer labels
        predicted: List of predicted answer strings (space-separated top 3)
    
    Returns:
        MAP@3 score
    """
    def ap_at_3(a, p):
        """Average Precision at 3 for single prediction"""
        p = p.split()
        score = 0.0
        num_hits = 0
        
        for i, pred in enumerate(p[:3]):
            if pred == a and pred not in p[:i]:
                num_hits += 1
                score += num_hits / (i + 1)
        
        return score
    
    return np.mean([ap_at_3(a, p) for a, p in zip(actual, predicted)])

In [ ]:
# Calculate train MAP@3
train_map3 = map_at_3(train['answer'].tolist(), train_predictions)
print(f"TRAIN MAP@3 (TF-IDF Baseline): {train_map3:.4f}")

In [ ]:
# Log to W&B
wandb.log({
    "train_map3": train_map3,
    "model": "TF-IDF Cosine Similarity",
    "vocab_size": len(tfidf.vocabulary_)
})